# Credit Card Fraud Detection — Exploratory Data Analysis

This notebook performs a thorough exploratory data analysis (EDA) of the public **Credit Card Fraud Detection** dataset (284,807 European cardholder transactions, 492 of which are fraudulent).

**Contents**
1. Load data & basic structure
2. Missing values & duplicates
3. Class distribution (fraud vs. legitimate)
4. Transaction amount distribution
5. Correlation heatmap
6. Outlier analysis (boxplots)
7. Distribution plots / histograms per class
8. Pair plot of most informative features

> **Note:** Run this notebook from the project root, and make sure `data/raw/creditcard.csv` exists (see `README.md` for download instructions).

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_raw_data
from src.preprocessing import check_missing_values, remove_duplicates

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
%matplotlib inline

## 1. Load data & basic structure

In [ ]:
df = load_raw_data()
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 2. Missing values & duplicates

This dataset is documented as having no missing values, but we verify this programmatically rather than assuming it — a core good practice for any real-world pipeline. We also check for and remove exact duplicate rows, which can otherwise leak information between train/test splits and bias metrics optimistically.

In [ ]:
missing = check_missing_values(df)
print('Columns with missing values:')
print(missing if not missing.empty else 'None')

In [ ]:
n_dupes = df.duplicated().sum()
print(f'Duplicate rows: {n_dupes} ({n_dupes / len(df) * 100:.3f}% of data)')
df_clean = remove_duplicates(df)
print('Shape after removing duplicates:', df_clean.shape)

## 3. Class distribution: Fraud vs. Non-Fraud

This is the single most important chart in the whole analysis: it shows just how extreme the class imbalance is, which drives every downstream modeling decision (resampling, choice of metric, threshold tuning).

In [ ]:
class_counts = df_clean['Class'].value_counts()
class_pct = df_clean['Class'].value_counts(normalize=True) * 100

print(class_counts)
print(class_pct.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.countplot(x='Class', data=df_clean, ax=axes[0], hue='Class', legend=False, palette=['#4C72B0', '#C44E52'])
axes[0].set_title('Transaction Count by Class (linear scale)')
axes[0].set_xticks([0, 1])
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])

sns.countplot(x='Class', data=df_clean, ax=axes[1], hue='Class', legend=False, palette=['#4C72B0', '#C44E52'])
axes[1].set_yscale('log')
axes[1].set_title('Transaction Count by Class (log scale)')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Legitimate (0)', 'Fraud (1)'])
plt.tight_layout()
plt.savefig('../outputs/class_distribution.png', dpi=150)
plt.show()

**Interpretation:** Fraudulent transactions make up roughly **0.17%** of the dataset. A naive model predicting 'legitimate' for every transaction would score ~99.83% accuracy while catching zero fraud — this is exactly why accuracy is the wrong metric here, and why class-imbalance handling (Section 3 of `feature_engineering.py`) and PR AUC / recall / precision (not just ROC AUC) matter so much for this problem.

## 4. Transaction amount distribution

We compare the distribution of `Amount` between legitimate and fraudulent transactions.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_clean[df_clean['Class'] == 0]['Amount'], bins=50, ax=axes[0], color='#4C72B0', kde=True)
axes[0].set_title('Amount Distribution — Legitimate')
axes[0].set_xlim(0, 2500)

sns.histplot(df_clean[df_clean['Class'] == 1]['Amount'], bins=50, ax=axes[1], color='#C44E52', kde=True)
axes[1].set_title('Amount Distribution — Fraud')
axes[1].set_xlim(0, 2500)

plt.tight_layout()
plt.savefig('../outputs/amount_distribution.png', dpi=150)
plt.show()

print(df_clean.groupby('Class')['Amount'].describe())

**Interpretation:** Fraudulent transactions tend to cluster at lower amounts with a long tail, and their median amount typically differs from legitimate transactions — but the distributions overlap substantially, meaning `Amount` alone is a weak fraud signal and must be combined with the anonymized `V1`-`V28` PCA features.

## 5. Correlation heatmap

Since `V1`-`V28` are already PCA components (and therefore mutually uncorrelated by construction), we expect near-zero inter-feature correlation among them, but we still check correlation with `Class` to identify which components carry the strongest fraud signal.

In [ ]:
corr = df_clean.corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(corr, cmap='coolwarm', center=0, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Heatmap — All Features')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
class_corr = corr['Class'].drop('Class').sort_values(key=abs, ascending=False)
print('Top 10 features most correlated (by magnitude) with Class:')
print(class_corr.head(10))

fig, ax = plt.subplots(figsize=(8, 6))
class_corr.head(10).plot(kind='barh', ax=ax, color='#55A868')
ax.set_title('Top 10 Features Correlated with Fraud (Class)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

**Interpretation:** Features such as `V17`, `V14`, `V12`, `V10` (exact ranking varies slightly by data version) tend to show the strongest linear correlation with the fraud label, consistent with their high ranking in the tree-based feature importance plots produced later by `evaluate.py`.

## 6. Outlier analysis (boxplots)

We inspect the top correlated features for outliers, split by class, since extreme values in these PCA components are often exactly what signals a fraudulent transaction.

In [ ]:
top_features = class_corr.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flatten(), top_features):
    sns.boxplot(x='Class', y=feature, data=df_clean, ax=ax, hue='Class', legend=False, palette=['#4C72B0', '#C44E52'])
    ax.set_title(f'{feature} by Class')
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Legit', 'Fraud'])
plt.tight_layout()
plt.savefig('../outputs/boxplots_top_features.png', dpi=150)
plt.show()

**Interpretation:** Fraudulent transactions show visibly wider interquartile ranges and more extreme outliers on several of the top correlated components — exactly the kind of separation tree-based models (Random Forest, Gradient Boosting, XGBoost) exploit effectively via axis-aligned splits.

## 7. Distribution plots / histograms per class

Overlaid histograms of the top features, split by class, to visually assess how separable each feature is.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for ax, feature in zip(axes.flatten(), top_features):
    sns.histplot(df_clean[df_clean['Class'] == 0][feature], bins=50, ax=ax, color='#4C72B0', label='Legit', stat='density', kde=True)
    sns.histplot(df_clean[df_clean['Class'] == 1][feature], bins=50, ax=ax, color='#C44E52', label='Fraud', stat='density', kde=True)
    ax.set_title(f'Distribution of {feature}')
    ax.legend()
plt.tight_layout()
plt.savefig('../outputs/histograms_top_features.png', dpi=150)
plt.show()

## 8. Pair plot of most informative features

A pair plot on the top 4 correlated features (subsampled for performance — pair plots on 284K rows are prohibitively slow and visually unreadable) shows how well fraud and legitimate transactions separate in a handful of dimensions simultaneously.

In [ ]:
pairplot_features = top_features[:4] + ['Class']

# Subsample: keep ALL fraud cases, sample a matching-size batch of legit
# transactions so the plot is both fast to render and visually balanced.
fraud_df = df_clean[df_clean['Class'] == 1]
legit_sample = df_clean[df_clean['Class'] == 0].sample(n=len(fraud_df) * 3, random_state=42)
pairplot_df = pd.concat([fraud_df, legit_sample])[pairplot_features]

g = sns.pairplot(pairplot_df, hue='Class', palette=['#4C72B0', '#C44E52'], diag_kind='kde', plot_kws={'alpha': 0.5, 's': 15})
g.fig.suptitle('Pair Plot of Top Correlated Features (fraud + 3x-sized legit sample)', y=1.02)
g.savefig('../outputs/pairplot_top_features.png', dpi=150)
plt.show()

**Interpretation:** Even in just two or three dimensions, fraudulent transactions (red) form visually distinct clusters relative to the bulk of legitimate transactions (blue) along several of the top components — a strong signal that a nonlinear model should be able to separate the classes well once class imbalance is addressed.

## Summary of EDA findings

- The dataset is extremely imbalanced (~0.17% fraud), which must be addressed via resampling and imbalance-aware metrics.
- No missing values were found; a small number of exact duplicate rows were identified and removed.
- `Amount` alone is a weak signal; several PCA components (`V17`, `V14`, `V12`, `V10`, and similar) show much stronger correlation with fraud.
- Fraudulent transactions show more extreme outlier behavior and visibly separable clusters in the top correlated components, supporting the use of tree-based ensemble models.

These findings directly motivate the preprocessing, feature engineering, and modeling choices implemented in `src/preprocessing.py`, `src/feature_engineering.py`, and `src/train.py`.